#  E-Waste Dataset Exploration

This notebook helps you:
- Understand the dataset structure and class distribution
- Visualize sample images with bounding boxes
- Analyze bounding box sizes and aspect ratios
- Preview Albumentations augmentations


In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

matplotlib.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'font.family':      'monospace',
})

from src.utils import CLASS_NAMES, CLASS_COLORS

# Dataset paths
DATASET_ROOT  = ROOT / 'dataset'
TRAIN_IMG_DIR = DATASET_ROOT / 'images' / 'train'
VAL_IMG_DIR   = DATASET_ROOT / 'images' / 'val'
TRAIN_LBL_DIR = DATASET_ROOT / 'labels' / 'train'
VAL_LBL_DIR   = DATASET_ROOT / 'labels' / 'val'

print(f'Project root : {ROOT}')
print(f'Train images : {TRAIN_IMG_DIR}')
print(f'Val images   : {VAL_IMG_DIR}')


## 1️  Dataset Statistics


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(folder: Path) -> int:
    return sum(1 for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS) if folder.exists() else 0

def count_labels(folder: Path) -> int:
    return sum(1 for f in folder.iterdir() if f.suffix == '.txt') if folder.exists() else 0

n_train = count_images(TRAIN_IMG_DIR)
n_val   = count_images(VAL_IMG_DIR)
n_lbl_train = count_labels(TRAIN_LBL_DIR)
n_lbl_val   = count_labels(VAL_LBL_DIR)

print('=' * 45)
print(f'  Split        | Images  | Labels')
print('=' * 45)
print(f'  Train        | {n_train:>6}  | {n_lbl_train:>6}')
print(f'  Validation   | {n_val:>6}  | {n_lbl_val:>6}')
print(f'  Total        | {n_train+n_val:>6}  | {n_lbl_train+n_lbl_val:>6}')
print('=' * 45)

if n_train == 0:
    print('\n⚠️  Training folder is empty.')
    print('   Add images and labels to dataset/images/train and dataset/labels/train.')


## 2️  Class Distribution


In [ ]:
def load_annotations(label_dir: Path) -> pd.DataFrame:
    """Load all YOLO .txt annotation files into a DataFrame."""
    rows = []
    if not label_dir.exists():
        return pd.DataFrame(columns=['file','class_id','xc','yc','w','h'])
    for txt in label_dir.glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cls, xc, yc, w, h = parts
                rows.append({'file': txt.stem, 'class_id': int(cls),
                             'xc': float(xc), 'yc': float(yc),
                             'w': float(w),   'h': float(h)})
    return pd.DataFrame(rows)

train_df = load_annotations(TRAIN_LBL_DIR)
val_df   = load_annotations(VAL_LBL_DIR)
all_df   = pd.concat([train_df, val_df], ignore_index=True)

if all_df.empty:
    print('No labels found. Add labels to dataset/labels/ first.')
else:
    class_counts = all_df['class_id'].value_counts().sort_index()
    bar_colors = [f'#{CLASS_COLORS[i][2]:02x}{CLASS_COLORS[i][1]:02x}{CLASS_COLORS[i][0]:02x}' 
                  for i in class_counts.index]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar([CLASS_NAMES[i] for i in class_counts.index], class_counts.values, color=bar_colors, edgecolor='none', linewidth=0)
    ax.set_title('Object Count per Class (all splits)', fontsize=13, pad=12)
    ax.set_xlabel('Class')
    ax.set_ylabel('Annotation Count')
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, class_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(v),
                ha='center', va='bottom', fontsize=9, color='#e6edf3')
    plt.tight_layout()
    plt.show()
    print(f'\nTotal annotations: {len(all_df)}')


## 3️  Sample Images with Bounding Boxes


In [ ]:
def draw_yolo_boxes(img: np.ndarray, label_path: Path, ax):
    """Draw YOLO-format bounding boxes on a Matplotlib axes."""
    h, w = img.shape[:2]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (xc - bw/2) * w
            y1 = (yc - bh/2) * h
            color_rgb = tuple(v/255 for v in CLASS_COLORS.get(cls, (180,180,180))[::-1])
            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                      linewidth=2, edgecolor=color_rgb, facecolor='none')
            ax.add_patch(rect)
            label = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else str(cls)
            ax.text(x1, y1 - 4, label, color=color_rgb, fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc='#0d1117', alpha=0.7, ec='none'))
    ax.axis('off')

sample_imgs = list(TRAIN_IMG_DIR.glob('*'))[:8] if TRAIN_IMG_DIR.exists() else []

if not sample_imgs:
    print('No training images found. Add images to dataset/images/train/')
else:
    n_show = min(8, len(sample_imgs))
    ncols = 4
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
    axes = axes.flatten()
    for i, img_path in enumerate(sample_imgs[:n_show]):
        img = cv2.imread(str(img_path))
        label_path = TRAIN_LBL_DIR / (img_path.stem + '.txt')
        draw_yolo_boxes(img, label_path, axes[i])
        axes[i].set_title(img_path.name[:25], fontsize=8, pad=3)
    for ax in axes[n_show:]:
        ax.axis('off')
    plt.suptitle('Sample Training Images', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


## 4️  Bounding Box Size Distribution


In [ ]:
if not all_df.empty:
    all_df['area'] = all_df['w'] * all_df['h']
    all_df['aspect_ratio'] = all_df['w'] / all_df['h']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(all_df['w'], bins=40, color='#00e5ff', edgecolor='none', alpha=0.85)
    axes[0].set_title('Box Width Distribution (normalized)')
    axes[0].set_xlabel('Normalized Width')
    axes[0].set_ylabel('Count')

    axes[1].hist(all_df['h'], bins=40, color='#76ff03', edgecolor='none', alpha=0.85)
    axes[1].set_title('Box Height Distribution (normalized)')
    axes[1].set_xlabel('Normalized Height')

    axes[2].hist(all_df['area'], bins=40, color='#ff6f00', edgecolor='none', alpha=0.85)
    axes[2].set_title('Box Area Distribution (w × h)')
    axes[2].set_xlabel('Normalized Area')

    for ax in axes:
        ax.grid(axis='y', alpha=0.25)

    plt.tight_layout()
    plt.show()

    print(all_df[['w','h','area','aspect_ratio']].describe().round(4))
else:
    print('No annotations available for distribution analysis.')


## 5️  Albumentations Augmentation Preview


In [ ]:
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    augment_pipeline = A.Compose([
        A.RandomBrightnessContrast(p=0.6),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.CoarseDropout(max_holes=6, max_height=32, max_width=32, p=0.3),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

    sample_imgs_list = list(TRAIN_IMG_DIR.glob('*'))[:4] if TRAIN_IMG_DIR.exists() else []

    if not sample_imgs_list:
        print('No training images found for augmentation preview.')
    else:
        n_aug = min(4, len(sample_imgs_list))
        fig, axes = plt.subplots(n_aug, 2, figsize=(12, n_aug * 4))
        if n_aug == 1:
            axes = [axes]

        for i, img_path in enumerate(sample_imgs_list[:n_aug]):
            img_bgr  = cv2.imread(str(img_path))
            img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            lbl_path = TRAIN_LBL_DIR / (img_path.stem + '.txt')
            bboxes, class_labels = [], []
            if lbl_path.exists():
                for line in lbl_path.read_text().strip().splitlines():
                    parts = line.split()
                    if len(parts) == 5:
                        class_labels.append(int(parts[0]))
                        bboxes.append([float(x) for x in parts[1:]])

            result = augment_pipeline(image=img_rgb, bboxes=bboxes, class_labels=class_labels)

            axes[i][0].imshow(img_rgb)
            axes[i][0].set_title(f'Original: {img_path.name[:20]}', fontsize=9)
            axes[i][0].axis('off')

            axes[i][1].imshow(result['image'])
            axes[i][1].set_title('Augmented', fontsize=9)
            axes[i][1].axis('off')

        plt.suptitle('Albumentations Augmentation Preview', fontsize=13, y=1.01)
        plt.tight_layout()
        plt.show()

except ImportError:
    print('albumentations not installed. Run: pip install albumentations')


---
##  Next Steps

1. **Add real images** to `dataset/images/train` and `dataset/images/val`
2. **Annotate** with [Roboflow](https://roboflow.com) and export in YOLO format
3. **Train** with: `python src/train.py --data data.yaml --epochs 50`
4. **Test** with: `python src/detect.py --source path/to/image.jpg --weights models/train/weights/best.pt`
5. **Run demo** with: `streamlit run app/streamlit_app.py`
